# HydroSovereign AI Engine — Quick Start Notebook

This notebook demonstrates how to use `hydrosovereign` to assess transboundary water law compliance for any river basin.

**Install:** `pip install hydrosovereign`
**DOI:** https://doi.org/10.5281/zenodo.19180160
**Author:** Seifeldin M.G. Alkhedir · ORCID: 0000-0003-0821-2991

In [ ]:
# Install if needed
# !pip install hydrosovereign[viz]

from hydrosovereign import ATDI, AHIFD, AFSF, AHLB, ASI, ATCI
from hydrosovereign import ConflictIndex, NegotiationAI
import pandas as pd

## 1. Analyse a Single Basin (Blue Nile / GERD)

In [ ]:
params_gerd = dict(
    runoff_coeff=0.38,
    dam_capacity_bcm=74.0,
    n_countries=3,
    dispute_level=4,
    basin_area_km2=174000
)

atdi  = ATDI(**params_gerd)
ahifd = AHIFD(**params_gerd)
atci  = ATCI(**params_gerd)
ci    = ConflictIndex(atdi=atdi, ahifd=ahifd, **params_gerd)

print(f'Blue Nile (GERD):')
print(f'  ATDI  = {atdi:.1f}%  → Art. 7 UNWC {"TRIGGERED" if atdi >= 40 else "OK"}')
print(f'  AHIFD = {ahifd:.1f}%  → {ahifd:.0f}% of natural flow withheld')
print(f'  ATCI  = {atci:.0f}/100')
print(f'  CI    = {ci:.3f}')

## 2. Multi-Basin Comparison (26 Basins)

In [ ]:
from hydrosovereign.basins import BASINS_26

results = []
for basin in BASINS_26:
    p = dict(
        runoff_coeff=basin.get('runoff_c', 0.35),
        dam_capacity_bcm=basin.get('cap_bcm', 10),
        n_countries=basin.get('n_countries', 3),
        dispute_level=basin.get('dispute_level', 2),
        basin_area_km2=basin.get('area_km2', 100000)
    )
    results.append({
        'Basin': basin['name'],
        'Region': basin.get('continent', '?'),
        'ATDI%': round(ATDI(**p), 1),
        'AHIFD%': round(AHIFD(**p), 1),
        'ATCI': round(ATCI(**p), 0),
    })

df = pd.DataFrame(results).sort_values('ATDI%', ascending=False)
print(df.to_string(index=False))

## 3. NegotiationAI Pathway Assessment

In [ ]:
ai = NegotiationAI()
basins_test = ['Blue Nile', 'Euphrates', 'Mekong', 'Colorado']

for name in basins_test:
    p_neg = ai.predict(atdi=45.0, ci=0.45, n_countries=3, dispute_level=3)
    print(f'{name:30s}  P(Negotiation)={p_neg:.0%}')